# HGRIA - Hand Gesture Recognition for Interactive Applications
## Launch Notebook

```
┌─────────────────────────────────────────────────────────────┐
│                    ARCHITECTURE                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   Browser (Frontend)     ngrok Tunnel      Colab Backend   │
│   ┌──────────────┐      ┌──────────┐     ┌──────────────┐   │
│   │  GitHub Pages │ ←─── │  HTTPS   │ ←── │  Flask +     │   │
│   │  / Vercel    │      │  Tunnel  │     │  MediaPipe   │   │
│   └──────────────┘      └──────────┘     └──────────────┘   │
│         │                                          │        │
│         │           Google Drive                    │        │
│         └──────────────┬───────────────────────────┘        │
│                        │ logs/                             │
└─────────────────────────────────────────────────────────────┘
```

### Prerequisites
- Google Account with Google Drive access
- ngrok account (free tier works)
- WebRTC-compatible browser (Chrome, Edge, Firefox)

### How it works
1. Backend runs Flask server on Colab with MediaPipe
2. ngrok creates HTTPS tunnel to expose backend
3. Frontend connects via WebSocket and sends webcam frames
4. Backend processes frames and sends gesture commands back

In [ ]:
# Step 1: Mount Google Drive and verify project files
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

SOURCE_PATH = '/content/drive/MyDrive/HGRIA'
REQUIREMENTS_FILE = os.path.join(SOURCE_PATH, 'requirements.txt')

if not os.path.exists(REQUIREMENTS_FILE):
    raise FileNotFoundError(
        f"requirements.txt not found at {REQUIREMENTS_FILE}.\n"
        "Please ensure HGRIA project is saved in your Google Drive at:\n"
        "  /content/drive/MyDrive/HGRIA/"
    )

print(f"✓ Project files found at {SOURCE_PATH}")

In [ ]:
# Step 2: Copy project to Colab and install dependencies
import shutil
import subprocess
import sys

DEST = '/content/HGRIA'

# Copy only if destination doesn't exist or user wants to refresh
if os.path.exists(DEST):
    print(f"Project already exists at {DEST}")
else:
    shutil.copytree(SOURCE_PATH, DEST)
    print(f"✓ Copied project to {DEST}")

# Add to Python path
sys.path.insert(0, DEST)
os.chdir(DEST)

# Install dependencies
result = subprocess.run(
    ['pip', 'install', '-q', '-r', 'requirements.txt'],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"Failed to install dependencies:\n{result.stderr}"
    )

print("✓ Dependencies installed successfully")

In [ ]:
# Step 3: Configure ngrok authentication
import getpass

# Install pyngrok if not present
subprocess.run(['pip', 'install', '-q', 'pyngrok'], capture_output=True)
from pyngrok import ngrok

# Get ngrok auth token (optional but recommended)
print("Enter your ngrok authtoken (from https://dashboard.ngrok.com/auth)")
print("Press Enter to skip (anonymous tunnel may disconnect)")

authtoken = getpass.getpass(prompt='Authtoken: ')

if authtoken:
    ngrok.set_auth_token(authtoken)
    print("✓ ngrok authenticated")
else:
    print("⚠ Anonymous tunnel - connection may be unstable")

In [ ]:
# Step 4: Start ngrok tunnel and display connection info
import json

# Start ngrok tunnel
tunnel = ngrok.connect(5000, "http")

# Convert HTTP URL to HTTPS
ngrok_url = tunnel.public_url.replace('http://', 'https://')

print("=" * 60)
print("🔗 NGROK TUNNEL READY")
print("=" * 60)
print(f"\nBackend URL: {ngrok_url}")

# Generate frontend integration snippet
frontend_script = f'<script>window.HGRIA_BACKEND_URL="{ngrok_url}";<\/script>'
print(f"\nPaste this in your frontend HTML (before Socket.IO loads):")
print(f"\n{frontend_script}")

# Generate frontend URL (assuming GitHub Pages or Vercel)
frontend_base = "https://your-username.github.io/HGRIA"
frontend_url = f"{frontend_base}?server={ngrok_url}"

print(f"\nOr open Frontend directly with:")
print(f"\n{frontend_url}")
print("\n" + "=" * 60)

In [ ]:
# Step 5: Configure and patch config for Colab mode
import json

config_path = os.path.join(DEST, 'config', 'config.json')

# Load existing config
with open(config_path, 'r') as f:
    config = json.load(f)

# Apply Colab-specific patches
config['camera']['colab_mode'] = True
config['server']['cors_origins'] = '*'
config['logging']['log_to_file'] = True
config['logging']['log_file_path'] = '/content/drive/MyDrive/HGRIA/logs/'

# Write patched config back
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Config patched for Colab mode:")
print(f"  - colab_mode: {config['camera']['colab_mode']}")
print(f"  - cors_origins: {config['server']['cors_origins']}")
print(f"  - log_to_file: {config['logging']['log_to_file']}")
print(f"  - log_file_path: {config['logging']['log_file_path']}")

In [ ]:
# Step 6: Start the HGRIA Server (blocking)
# This cell will keep running until interrupted
from backend.main import SystemOrchestrator

print("Starting HGRIA Backend Server...")
print(f"Server running at: {ngrok_url}")
print("\nPress Stop button to terminate the server")
print("-" * 40)

# Start the orchestrator (blocking)
orchestrator = SystemOrchestrator(config_path)
orchestrator.start()

## Post-Launch Instructions

### Accessing the Frontend

After the server starts, open your browser and navigate to:

```
https://your-username.github.io/HGRIA/?server=<NGROK_URL>
```

Or paste the `window.HGRIA_BACKEND_URL` script into the frontend HTML before Socket.IO.

### If ngrok URL Changes

1. Stop the server (interrupt the cell above)
2. Re-run cells 4, 5, and 6 in sequence
3. Update the frontend with the new URL

### Troubleshooting

| Issue | Solution |
|-------|----------|
| Colab session timeout | Re-run cell 6 (server restart is automatic) |
| ngrok URL changed | Re-run cells 4, 5, 6 and update frontend |
| Webcam denied | Use keyboard fallback (Arrow keys, Space, P, S) |
| High latency | Check Colab GPU availability |

### Keyboard Controls (Fallback)
- Arrow Keys: Move
- Space: Jump
- P: Pause
- S: Speed Boost
- Enter: Confirm